# Preços de combustíveis no Brasil — análise exploratória (2015–2024)

**Projeto G2 • Tema 11 • Análise e Visualização de Dados com Python**

Este notebook investiga o comportamento dos preços de gasolina, diesel, etanol, GNV e GLP no Brasil entre 2015 e 2024, usando a base simulada `simulacao_precos_combustiveis_brasil.csv` disponibilizada pelo professor.

## 1. Introdução ao problema

O preço dos combustíveis é uma variável sensível da economia brasileira. Ele afeta o custo de transporte urbano e de cargas, pressiona a inflação (os combustíveis têm peso relevante no IPCA), e muda a competitividade entre modais e entre combustíveis substitutos (gasolina × etanol, diesel × GNV).

Entre 2015 e 2024 o país atravessou vários choques: queda do petróleo em 2015–2016, mudança na política de preços da Petrobras, greve dos caminhoneiros em 2018, pandemia em 2020, pico do Brent em 2022 e a volta de uma política de suavização em 2023–2024. Investigar como preços se comportaram ao longo desse período é um exercício clássico de análise exploratória.

**Perguntas que o notebook tenta responder:**

1. Como os preços evoluíram ao longo do tempo?
2. Quais estados e regiões apresentam preços mais altos e mais baixos?
3. Qual combustível teve maior aumento no período?
4. Qual combustível é mais volátil?
5. Existe correlação entre preço, inflação e cotação do petróleo?

## 2. Explicação da base de dados

A base é **simulada** e cobre o período de janeiro de 2015 a dezembro de 2024, com observações mensais por UF e combustível. As colunas são:

| coluna | descrição |
|---|---|
| `ano`, `mes`, `data` | referência temporal da observação |
| `regiao`, `uf` | localização geográfica (5 regiões, 20 UFs representadas) |
| `combustivel` | Gasolina, Diesel, Etanol, GNV, GLP |
| `preco_medio`, `preco_minimo`, `preco_maximo` | preços em R$/L naquele mês/UF |
| `variacao_mensal` | variação % em relação ao mês anterior |
| `inflacao` | indicador mensal de inflação |
| `cotacao_petroleo` | cotação do petróleo (US$) |
| `consumo_estimado` | consumo estimado no mês |
| `nivel_preco` | classificação qualitativa (Baixo / Médio / Alto / Crítico) |

> Por ser uma base simulada, as relações entre as variáveis econômicas não refletem necessariamente o comportamento histórico real — este ponto será retomado na interpretação dos resultados.

## 3. Leitura dos dados

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid', rc={'axes.spines.top': False, 'axes.spines.right': False})
plt.rcParams['figure.dpi'] = 110

CSV = Path('..') / 'dados' / 'simulacao_precos_combustiveis_brasil.csv'
df = pd.read_csv(CSV)
print('Shape:', df.shape)
df.head()

In [ ]:
df.info()

## 4. Limpeza e preparação

A base chega limpa, sem nulos e com tipos razoáveis. As preparações necessárias são apenas: converter `data` para `datetime`, garantir `ano`/`mes` como inteiros e checar unicidade da chave (data × uf × combustível).

In [ ]:
print('Nulos por coluna:')
print(df.isna().sum())
print('\nDuplicatas na chave (data, uf, combustível):', df.duplicated(subset=['data','uf','combustivel']).sum())

In [ ]:
df['data'] = pd.to_datetime(df['data'])
df['ano'] = df['ano'].astype(int)
df['mes'] = df['mes'].astype(int)

print('Período:', df['data'].min().date(), 'a', df['data'].max().date())
print('Regiões:', df['regiao'].unique())
print('UFs:', sorted(df['uf'].unique()))
print('Combustíveis:', df['combustivel'].unique())
print('Níveis:', df['nivel_preco'].unique())

## 5. Engenharia de atributos

Alguns campos derivados ajudam nas análises temporais e comparativas:

- `trimestre` — para agregações sazonais
- `amplitude` — `preco_maximo − preco_minimo`
- `indice_base100` — preço indexado para cada combustível (base = primeiro mês)

In [ ]:
df['trimestre'] = df['data'].dt.quarter
df['amplitude'] = df['preco_maximo'] - df['preco_minimo']

serie_comb = df.groupby(['data', 'combustivel'], as_index=False)['preco_medio'].mean()
serie_comb = serie_comb.sort_values(['combustivel', 'data'])
serie_comb['indice_base100'] = (
    serie_comb.groupby('combustivel')['preco_medio']
    .transform(lambda s: s / s.iloc[0] * 100)
)
serie_comb.head()

## 6. Análise exploratória

In [ ]:
df[['preco_medio', 'preco_minimo', 'preco_maximo', 'variacao_mensal',
    'inflacao', 'cotacao_petroleo', 'consumo_estimado']].describe().round(2)

In [ ]:
resumo = df.groupby('combustivel').agg(
    media=('preco_medio', 'mean'),
    mediana=('preco_medio', 'median'),
    desvio=('preco_medio', 'std'),
    minimo=('preco_minimo', 'min'),
    maximo=('preco_maximo', 'max'),
).round(2).sort_values('media', ascending=False)
resumo

## 7. KPIs principais

In [ ]:
serie = df.groupby('data')['preco_medio'].mean().sort_index()
variacao_total = (serie.iloc[-1] / serie.iloc[0] - 1) * 100

kpis = {
    'Preço médio geral (R$/L)': round(df['preco_medio'].mean(), 2),
    'Preço mínimo observado': round(df['preco_minimo'].min(), 2),
    'Preço máximo observado': round(df['preco_maximo'].max(), 2),
    'Volatilidade (desvio)': round(df['preco_medio'].std(), 2),
    'Variação ponta-a-ponta (%)': round(variacao_total, 1),
    'Combustível mais caro (média)': df.groupby('combustivel')['preco_medio'].mean().idxmax(),
    'UF mais cara': df.groupby('uf')['preco_medio'].mean().idxmax(),
    'UF mais barata': df.groupby('uf')['preco_medio'].mean().idxmin(),
    'Região mais cara': df.groupby('regiao')['preco_medio'].mean().idxmax(),
}
pd.Series(kpis, name='valor').to_frame()

## 8. Visualizações

### 8.1 Evolução temporal por combustível

In [ ]:
paleta = {'Gasolina': '#e85d04', 'Diesel': '#6a4c93', 'Etanol': '#2a9d8f',
          'GNV': '#118ab2', 'GLP': '#d62828'}

fig, ax = plt.subplots(figsize=(11, 5))
for comb, grupo in serie_comb.groupby('combustivel'):
    ax.plot(grupo['data'], grupo['preco_medio'], label=str(comb),
            color=paleta.get(str(comb), '#333'), linewidth=1.6)
ax.set_title('Preço médio mensal por combustível — 2015 a 2024')
ax.set_ylabel('R$/L')
ax.set_xlabel('')
ax.legend(frameon=False, ncol=5)
plt.tight_layout()
plt.show()

### 8.2 Índice base 100 — variação acumulada

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))
for comb, grupo in serie_comb.groupby('combustivel'):
    ax.plot(grupo['data'], grupo['indice_base100'], label=str(comb),
            color=paleta.get(str(comb), '#333'), linewidth=1.6)
ax.axhline(100, color='#999', linewidth=0.8, linestyle='--')
ax.set_title('Preço indexado — base 100 em janeiro de 2015')
ax.set_ylabel('índice')
ax.legend(frameon=False, ncol=5)
plt.tight_layout()
plt.show()

### 8.3 Comparação regional

In [ ]:
pivot = df.pivot_table(index='regiao', columns='combustivel',
                       values='preco_medio', aggfunc='mean').round(2)

fig, ax = plt.subplots(figsize=(9, 4))
sns.heatmap(pivot, annot=True, fmt='.2f', cmap='YlOrRd',
            cbar_kws={'label': 'R$/L'}, ax=ax)
ax.set_title('Preço médio (R$/L) por região × combustível')
plt.tight_layout()
plt.show()

### 8.4 Dispersão por combustível

In [ ]:
ordem = sorted(df['combustivel'].unique())
fig, ax = plt.subplots(figsize=(9, 4.5))
sns.boxplot(data=df, x='combustivel', y='preco_medio',
            order=ordem, palette=[paleta.get(c, '#333') for c in ordem], ax=ax)
ax.set_title('Distribuição do preço por combustível')
ax.set_xlabel(''); ax.set_ylabel('R$/L')
plt.tight_layout()
plt.show()

### 8.5 Ranking de UFs

In [ ]:
por_uf = df.groupby('uf')['preco_medio'].mean().sort_values()
fig, ax = plt.subplots(figsize=(7, 7))
ax.barh(por_uf.index.astype(str), por_uf.values, color='#264653')
ax.set_title('Preço médio por UF (2015–2024)')
ax.set_xlabel('R$/L')
for i, v in enumerate(por_uf.values):
    ax.text(v, i, f' {v:.2f}', va='center', fontsize=8)
plt.tight_layout()
plt.show()

### 8.6 Correlação com inflação e petróleo

In [ ]:
agg = df.groupby('data').agg(
    preco=('preco_medio', 'mean'),
    inflacao=('inflacao', 'mean'),
    petroleo=('cotacao_petroleo', 'mean'),
    consumo=('consumo_estimado', 'mean'),
)
corr = agg.corr().round(3)

fig, ax = plt.subplots(figsize=(5.5, 4))
sns.heatmap(corr, annot=True, cmap='RdBu_r', vmin=-1, vmax=1, center=0, ax=ax)
ax.set_title('Matriz de correlação (séries mensais)')
plt.tight_layout()
plt.show()
corr

## 9. Interpretação dos resultados

**O que a análise mostra na base simulada:**

- Os cinco combustíveis têm preços médios muito próximos entre si (em torno de R$ 5,50/L) e distribuições parecidas, com desvios na mesma faixa — não há um combustível claramente mais caro ou mais volátil.
- As diferenças regionais são pequenas: a amplitude entre a região mais cara e a mais barata não passa de poucos centavos.
- A variação ponta-a-ponta (2015 → 2024) é modesta e varia por combustível, sem o padrão forte de alta que caracterizaria uma série real da ANP nesse período.
- A matriz de correlação entre preço médio, inflação e cotação do petróleo fica próxima de zero.

**Por que esse resultado faz sentido:** a base é **simulada** e os preços foram gerados em uma faixa uniforme (aproximadamente R$ 3–R$ 8), sem incorporar os choques, os reajustes e a estrutura de causa-e-efeito do mercado real. Em uma série verdadeira da ANP, veríamos:

- gasolina acompanhando movimentos do Brent com defasagem de algumas semanas,
- picos claros em 2018 (greve dos caminhoneiros) e 2022 (invasão da Ucrânia + câmbio),
- diferenças regionais estruturais (preço mais alto no Norte por logística),
- correlação positiva relevante entre preço do diesel e cotação do petróleo.

O exercício, portanto, demonstra o **pipeline analítico completo**, pronto para ser reaplicado sobre dados reais da [Série Histórica da ANP](https://www.gov.br/anp/pt-br/centrais-de-conteudo/dados-abertos/serie-historica-de-precos-de-combustiveis).

## 10. Conclusão

O projeto cumpriu as etapas de uma análise exploratória completa: carga, limpeza, engenharia de atributos, cálculo de KPIs, visualizações temporais e comparativas, matriz de correlação, e interpretação crítica dos resultados.

O dashboard Streamlit complementa o notebook oferecendo exploração interativa com filtros múltiplos (ano, mês, região, UF, combustível, nível), KPIs dinâmicos, persistência em SQLite via SQLAlchemy, upload de CSV alternativo e consulta SQL ao vivo — atendendo os critérios obrigatórios e avançados do enunciado.

O limite principal é a natureza simulada da base: ela permite demonstrar as técnicas, mas não gera leitura econômica substantiva. O próximo passo natural é apontar o mesmo pipeline para a série real da ANP, mantendo a estrutura do notebook e do app praticamente inalterada.